In [2]:
from datetime import datetime
import sqlite3
import pandas as pd


class UPITransactionAnalyzer:
    """A data processing pipeline to ingest, clean, and run quantitative

    analytics on UPI payment datasets using pandas and in-memory SQLite.
    """

    def __init__(self, dataset_path: str):
        self.file_path = dataset_path
        self.df = pd.DataFrame()

    def ingest_and_sanitize(self) -> "UPITransactionAnalyzer":
        """Reads CSV data and normalizes schemas dynamically."""
        try:
            raw_data = pd.read_csv(self.file_path)
            raw_data.columns = [col.strip() for col in raw_data.columns]

            # Filter out non-successful payments if status field exists
            status_fields = [
                c for c in raw_data.columns if "status" in c.lower()
            ]
            if status_fields:
                s_col = status_fields[0]
                raw_data = raw_data[
                    raw_data[s_col].astype(str).str.strip().str.upper()
                    == "SUCCESS"
                ]

            # Infer mandatory columns (amount & timestamp)
            amt_target = next(
                (
                    c
                    for c in raw_data.columns
                    if any(k in c.lower() for k in ["amount", "val", "price"])
                ),
                None,
            )
            date_target = next(
                (
                    c
                    for c in raw_data.columns
                    if any(k in c.lower() for k in ["date", "time", "stamp"])
                ),
                None,
            )
            cat_target = next(
                (
                    c
                    for c in raw_data.columns
                    if any(
                        k in c.lower()
                        for k in ["cat", "type", "merchant", "vendor"]
                    )
                ),
                None,
            )

            if not amt_target or not date_target:
                raise KeyError(
                    "Unable to map required fields (Amount/Date) from CSV header."
                )

            # Drop missing essential fields
            clean_df = raw_data.dropna(subset=[amt_target, date_target]).copy()

            # Format fields
            clean_df["Amount"] = pd.to_numeric(
                clean_df[amt_target], errors="coerce"
            )
            clean_df["Date"] = (
                pd.to_datetime(clean_df[date_target], errors="coerce")
            ).dt.strftime("%Y-%m-%d")

            clean_df["Category"] = (
                clean_df[cat_target] if cat_target else "General Retail"
            )

            self.df = clean_df[["Date", "Amount", "Category"]].dropna()
            print(
                f"[SUCCESS] Processed {len(self.df)} valid transaction records."
            )

        except Exception as err:
            print(f"[ERROR] Data ingestion failed: {err}")
            self.df = pd.DataFrame()

        return self

    def execute_analytics(self) -> None:
        """Executes aggregate queries over the cleaned transaction table."""
        if self.df.empty:
            print("[WARN] Aborting analysis: Empty dataset.")
            return

        # Context manager handles database lifecycle automatically
        with sqlite3.connect(":memory:") as db_conn:
            self.df.to_sql(
                "upi_ledger", db_conn, index=False, if_exists="replace"
            )

            print("\n" + "=" * 55)
            print(" METRIC 1: MONTHLY VOLUME & TICKET SIZE")
            print("=" * 55)

            sql_monthly = """
                SELECT
                    strftime('%Y-%m', Date) AS transaction_month,
                    COUNT(1) AS tx_volume,
                    ROUND(SUM(Amount), 2) AS gross_spend_inr,
                    ROUND(AVG(Amount), 2) AS mean_ticket_size
                FROM upi_ledger
                GROUP BY 1
                ORDER BY 1 ASC;
            """
            monthly_metrics = pd.read_sql_query(sql_monthly, db_conn)
            print(monthly_metrics.to_string(index=False))

            print("\n" + "=" * 55)
            print(" METRIC 2: MERCHANT CATEGORY SPEND SHARE")
            print("=" * 55)

            sql_category = """
                WITH TotalCapital AS (
                    SELECT SUM(Amount) AS total_val FROM upi_ledger
                )
                SELECT
                    Category AS merchant_category,
                    COUNT(1) AS tx_count,
                    ROUND(SUM(Amount), 2) AS total_spend_inr,
                    ROUND((SUM(Amount) * 100.0 / TotalCapital.total_val), 2) AS portfolio_share_pct
                FROM upi_ledger, TotalCapital
                GROUP BY merchant_category
                ORDER BY total_spend_inr DESC;
            """
            category_metrics = pd.read_sql_query(sql_category, db_conn)
            print(category_metrics.to_string(index=False))


if __name__ == "__main__":
    pipeline = UPITransactionAnalyzer("upi_data.csv")
    pipeline.ingest_and_sanitize().execute_analytics()

[SUCCESS] Processed 502 valid transaction records.

 METRIC 1: MONTHLY VOLUME & TICKET SIZE
transaction_month  tx_volume  gross_spend_inr  mean_ticket_size
          2024-06        445       2255398.51           5068.31
          2024-07         57        284159.18           4985.25

 METRIC 2: MERCHANT CATEGORY SPEND SHARE
merchant_category  tx_count  total_spend_inr  portfolio_share_pct
   General Retail       502       2539557.69                100.0
